# **Unlock the Potential of PySpark DataFrames: Hands-on Tips and Personalizations**

`University of East London, Docklands Campus, 2023-24`

`module name`: **`Machine Learning on Big Data (CN7030) - MSc AI&DS`**

`Author`: **`Dr Amin Karami (PG Academic Lead in CDT School)`**

`E`: **`a.karami@uel.ac.uk`**

`W`: **`http://www.aminkarami.com/`**

---

**DataFrame (DF)**: Schema (named columns) + declarative language. A DataFrame is a Dataset organized into named columns. It is conceptually equivalent to a table in a relational database. It is very efficient for strucutred data.

data: https://drive.google.com/file/d/1HiP_TkWYClAmzhhOFzhXdOXTfZb9DoB_/view?usp=drive_link (641MB)

source: https://spark.apache.org/docs/latest/sql-programming-guide.html

source: https://spark.apache.org/docs/latest/api/python/reference/

# **Section 1: Initialize PySpark**

In [ ]:
!pip3 install pyspark

# **System Check**

# **Linking with Spark**

In [ ]:
# Linking with Spark
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
                    .appName("Tutorial2_CN7030") \
                    .master("local[2]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()

spark

# **Connect to the Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Section 2: Create PySpark DataFrame from CSV**

In [ ]:
df = spark.read.format('csv').load('/content/drive/MyDrive/data.csv', inferSchema = True, header = True)

# show table
df.show(truncate = True)

# show schema
df.printSchema()

# some info
print(df.count())
print(len(df.columns))


+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+
|                  id|               age|salary|             score|sales|height|weight|gender|category|income|expenses|
+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+
|  0.8017532427858894| 79.71301351894658|     4| 37.10369234532471|   47|    62|     5|Female|       B|     8|      85|
|  0.6565552949992319| 24.22600602366628|     6| 79.75769828959885|    8|    30|    42|  Male|       A|    70|      85|
|  0.2515595782593636| 70.97364852287149|    17| 71.71375356646551|   82|     1|    72|  Male|       A|    66|      53|
|  0.2073428376111074| 45.09378549789149|    32|              NULL|   59|    12|    21|  Male|       B|    56|       2|
|  0.6392921379278927|30.357970527906065|    13|              NULL|   22|    69|    58|Female|       A|     0|      74|
|  0.8505582285081454|              NULL

# How many partitions?

In [ ]:
df.rdd.getNumPartitions()

6

# Let's increase the number of partitions

In [ ]:
df2 = df.repartition(8)
df2.rdd.getNumPartitions()

8

In [ ]:
df3 = df.repartition('category')
df3.rdd.getNumPartitions()

2

In [ ]:
df4 = df.repartition(6, 'category')
df4.rdd.getNumPartitions()

6

In [ ]:
df5 = df.repartitionByRange(10, "income") # joins and aggregations
df5.rdd.getNumPartitions()

10

# Write the DF to disk in a partitioned manner

In [ ]:
df5.write.option('header', True) \
         .partitionBy("gender") \
         .mode("overwrite") \
         .csv("gender_df5")

In [ ]:
df4.write.option('header', True) \
         .partitionBy("gender") \
         .mode("overwrite") \
         .csv("gender_df4")

In [ ]:
df5.repartition(3).write.option('header', True) \
         .partitionBy("gender") \
         .mode("overwrite") \
         .csv("gender_df5_repartiton")

In [ ]:
df4.write.option('header', True) \
         .partitionBy("gender") \
         .mode("overwrite") \
         .csv("gender_df4")

# **Section 3: DataFrame Operations and Transformations**

In [ ]:
# SELECT
df4.select("age", "salary", "gender").show(5)

+------------------+------+------+
|               age|salary|gender|
+------------------+------+------+
| 24.22600602366628|     6|  Male|
| 70.97364852287149|    17|  Male|
|30.357970527906065|    13|Female|
| 57.94047153077633|    37|Female|
|              NULL|     0|Female|
+------------------+------+------+
only showing top 5 rows



In [ ]:
# FILTER
df4.filter((df4.age > 30) & (df4.salary>50)).show(5)

+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+
|                  id|               age|salary|             score|sales|height|weight|gender|category|income|expenses|
+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+
|0.054599328832547256|58.445521939730064|    84|29.619007900937078|   94|    72|    74|  Male|       A|    78|      87|
|  0.6455441593808561| 68.47061987702949|    75|              NULL|   62|    31|    44|Female|       A|     2|      42|
|  0.8316027808351703|   64.751100030111|    96|              NULL|   13|    24|    30|Female|       A|     5|      19|
| 0.28862497348283567| 34.91854369238886|    76|              NULL|   73|    43|    48|  Male|       A|    83|      39|
|  0.7408143015309203| 52.83048441274324|    95|46.769693535768866|   59|     4|    10|Female|       A|    19|      38|
+--------------------+------------------

In [ ]:
from pyspark.sql.functions import mean, sum, round

df4.groupBy("category").agg(
                            round(mean('salary'), 2).alias("mean_salary"),
                            sum('salary').alias("sum_salary"),
                            ).show(10)

+--------+-----------+----------+
|category|mean_salary|sum_salary|
+--------+-----------+----------+
|       A|      49.49| 247412615|
|       B|      49.51| 247582007|
+--------+-----------+----------+



In [ ]:
# SORT
df4.sort('height', 'score', ascending = False).show(10)

+--------------------+------------------+------+-----------------+-----+------+------+------+--------+------+--------+
|                  id|               age|salary|            score|sales|height|weight|gender|category|income|expenses|
+--------------------+------------------+------+-----------------+-----+------+------+------+--------+------+--------+
| 0.16933902279650814|              NULL|    37|99.99842061516966|   63|    99|    78|Female|       B|    60|      36|
|  0.4414179206080756| 47.31491599580799|    17|99.99629934376922|   96|    99|    53|Female|       A|    13|      63|
|  0.6365465779061935| 75.97020871674081|    59|99.99605629873773|   56|    99|    82|  Male|       A|    36|      64|
|   0.137572774067943| 46.72427635576165|    16|99.99478454051689|   14|    99|    68|Female|       B|    20|      47|
|  0.7188334449042442|              NULL|    24|99.99452559017527|    5|    99|    66|Female|       B|    73|      92|
|  0.6186372186898206| 68.40377455302632|     9|

In [ ]:
# rename a column
renamed_df = df4.withColumnRenamed("income", "annual_income")\
                .withColumnRenamed("score", "annual_score")
renamed_df.show(10)

+-------------------+------------------+------+------------------+-----+------+------+------+--------+-------------+--------+
|                 id|               age|salary|      annual_score|sales|height|weight|gender|category|annual_income|expenses|
+-------------------+------------------+------+------------------+-----+------+------+------+--------+-------------+--------+
| 0.6565552949992319| 24.22600602366628|     6| 79.75769828959885|    8|    30|    42|  Male|       A|           70|      85|
| 0.2515595782593636| 70.97364852287149|    17| 71.71375356646551|   82|     1|    72|  Male|       A|           66|      53|
| 0.6392921379278927|30.357970527906065|    13|              NULL|   22|    69|    58|Female|       A|            0|      74|
| 0.7555506990689408| 57.94047153077633|    37| 99.05156945373632|   18|    67|    94|Female|       A|           91|      89|
|0.34380469538701885|              NULL|     0| 68.12994102604317|   50|    66|    29|Female|       A|           39|  

In [ ]:
# drop
reduced_df = df4.drop("id", "height")
reduced_df.show(10)

+------------------+------+------------------+-----+------+------+--------+------+--------+
|               age|salary|             score|sales|weight|gender|category|income|expenses|
+------------------+------+------------------+-----+------+------+--------+------+--------+
| 24.22600602366628|     6| 79.75769828959885|    8|    42|  Male|       A|    70|      85|
| 70.97364852287149|    17| 71.71375356646551|   82|    72|  Male|       A|    66|      53|
|30.357970527906065|    13|              NULL|   22|    58|Female|       A|     0|      74|
| 57.94047153077633|    37| 99.05156945373632|   18|    94|Female|       A|    91|      89|
|              NULL|     0| 68.12994102604317|   50|    29|Female|       A|    39|      10|
| 78.00738662272417|    44|57.377461450339794|   32|    68|  Male|       A|     3|      35|
| 5.739474360847097|    60|              NULL|   69|    59|Female|       A|    89|      94|
|29.232534249025754|    33| 72.17303870051973|   19|     5|Female|       A|     

# **Section 4: Working with Missing Data**

In [ ]:
from pyspark.sql.functions import col

missing_values = df4.select([sum(col(c).isNull().cast("int")).alias(c) for c in df4.columns])
missing_values.show()

+---+-------+------+-------+-----+------+------+------+--------+------+--------+
| id|    age|salary|  score|sales|height|weight|gender|category|income|expenses|
+---+-------+------+-------+-----+------+------+------+--------+------+--------+
|  0|4000499|     0|4001316|    0|     0|     0|     0|       0|     0|       0|
+---+-------+------+-------+-----+------+------+------+--------+------+--------+



In [ ]:
# dropna()
df_copy = df4.select("*") # a logical copy that not affect df4
df_cleaned = df_copy.dropna()
df_cleaned.count()

3598970

In [ ]:
# drop selected columns
columns_to_check = ["age", "salary"]
df_copy = df4.select("*") # a logical copy that not affect df4
df_cleaned = df_copy.dropna(subset = columns_to_check)
df_cleaned.count()

5999501

In [ ]:
# Fill missing values with a constant value
df_copy = df4.select("*")
df_filled = df_copy.fillna({'age': 30, 'score': 50})
df_filled.show(40)

+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+
|                  id|               age|salary|             score|sales|height|weight|gender|category|income|expenses|
+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+
|  0.6565552949992319| 24.22600602366628|     6| 79.75769828959885|    8|    30|    42|  Male|       A|    70|      85|
|  0.2515595782593636| 70.97364852287149|    17| 71.71375356646551|   82|     1|    72|  Male|       A|    66|      53|
|  0.6392921379278927|30.357970527906065|    13|              50.0|   22|    69|    58|Female|       A|     0|      74|
|  0.7555506990689408| 57.94047153077633|    37| 99.05156945373632|   18|    67|    94|Female|       A|    91|      89|
| 0.34380469538701885|              30.0|     0| 68.12994102604317|   50|    66|    29|Female|       A|    39|      10|
| 0.07531261247891552| 78.00738662272417

In [ ]:
# Fill missing values with a stat (mean, median, mode, constant)
from pyspark.ml.feature import Imputer

imputer = Imputer(inputCols = ['age', 'score'],
                  outputCols = ['age_imputed', 'score_imputed'],
                  strategy = 'mean')
df_imputed = imputer.fit(df4).transform(df4)
df_imputed.show(40)

+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+------------------+------------------+
|                  id|               age|salary|             score|sales|height|weight|gender|category|income|expenses|       age_imputed|     score_imputed|
+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+------------------+------------------+
|  0.6565552949992319| 24.22600602366628|     6| 79.75769828959885|    8|    30|    42|  Male|       A|    70|      85| 24.22600602366628| 79.75769828959885|
|  0.2515595782593636| 70.97364852287149|    17| 71.71375356646551|   82|     1|    72|  Male|       A|    66|      53| 70.97364852287149| 71.71375356646551|
|  0.6392921379278927|30.357970527906065|    13|              NULL|   22|    69|    58|Female|       A|     0|      74|30.357970527906065| 49.99490561227385|
|  0.7555506990689408| 57.94047153077633|    37| 99.

In [ ]:
from pyspark.sql.functions import format_number, col

df_imputed = df_imputed.withColumn("age_imputed",format_number(col("age_imputed"), 0))
df_imputed = df_imputed.withColumn("score_imputed",format_number(col("score_imputed"), 3))

df_imputed.show(30)

+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+-----------+-------------+
|                  id|               age|salary|             score|sales|height|weight|gender|category|income|expenses|age_imputed|score_imputed|
+--------------------+------------------+------+------------------+-----+------+------+------+--------+------+--------+-----------+-------------+
|  0.6565552949992319| 24.22600602366628|     6| 79.75769828959885|    8|    30|    42|  Male|       A|    70|      85|         24|       79.758|
|  0.2515595782593636| 70.97364852287149|    17| 71.71375356646551|   82|     1|    72|  Male|       A|    66|      53|         71|       71.714|
|  0.6392921379278927|30.357970527906065|    13|              NULL|   22|    69|    58|Female|       A|     0|      74|         30|       49.995|
|  0.7555506990689408| 57.94047153077633|    37| 99.05156945373632|   18|    67|    94|Female|       A|    91|      89|     

In [ ]:
# advanced: replace missing values with a custom logic using UDF (user-defined functions)
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def custom_fill(value):
  if value is None:
    return "unknown"
  else:
    return value


custom_fill_udf = udf(custom_fill, StringType())
df_filled = df4.withColumn("score_filled", custom_fill_udf(df4.score))

df_filled.show(5)

+-------------------+------------------+------+-----------------+-----+------+------+------+--------+------+--------+-----------------+
|                 id|               age|salary|            score|sales|height|weight|gender|category|income|expenses|     score_filled|
+-------------------+------------------+------+-----------------+-----+------+------+------+--------+------+--------+-----------------+
| 0.6565552949992319| 24.22600602366628|     6|79.75769828959885|    8|    30|    42|  Male|       A|    70|      85|79.75769828959885|
| 0.2515595782593636| 70.97364852287149|    17|71.71375356646551|   82|     1|    72|  Male|       A|    66|      53|71.71375356646551|
| 0.6392921379278927|30.357970527906065|    13|             NULL|   22|    69|    58|Female|       A|     0|      74|          unknown|
| 0.7555506990689408| 57.94047153077633|    37|99.05156945373632|   18|    67|    94|Female|       A|    91|      89|99.05156945373632|
|0.34380469538701885|              NULL|     0|6